In [20]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve
import joblib

In [21]:
import os

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
    torch.set_num_threads(os.cpu_count())  # sfrutto tutti i core CPU per velocizzare il training

print(f"Sto addestrando su: {device}")

Sto addestrando su: cpu


In [22]:
#@title Caricamento e preparazione dati
df = pd.read_csv('AI_machine_learning_data_set.csv')
    
# Tolgo le colonne che non servono al modello (identificativi e flag non predittivi)
df_model = df.drop(columns=['nameOrig', 'nameDest', 'isFlaggedFraud', 'step'])

# Feature ingegnerizzate: differenze contabili, molto indicative di frode nel dataset PaySim
# (le frodi lasciano spesso un'incongruenza tra il saldo che ci si aspetterebbe dopo
# la transazione e quello effettivamente registrato)
df_model['errorBalanceOrig'] = df_model['oldbalanceOrg'] - df_model['amount'] - df_model['newbalanceOrig']
df_model['errorBalanceDest'] = df_model['oldbalanceDest'] + df_model['amount'] - df_model['newbalanceDest']

# Separo le feature (X) dalla variabile target (y)
X = df_model.drop('isFraud', axis=1)

y = df_model['isFraud'] 

# Divido in training (60%), validation (20%) e test (20%), mantenendo stratificata
# la percentuale di frodi in ogni set. Validation e test restano intatti e sbilanciati,
# per valutare il modello in condizioni realistiche.
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, stratify=y, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42)

# UNDER-SAMPLING MODERATO (solo sul training set)
# Usare tutte le ~3,8M righe di training rende ogni epoca troppo lenta. Invece di
# bilanciare 1:1 (versione precedente, che buttava via troppa informazione) tengo
# tutte le frodi + un multiplo di transazioni lecite (rapporto 1:20). Lo sbilanciamento
# residuo viene comunque corretto con pos_weight nella loss (vedi cella di configurazione).
RATIO_LECITE_PER_FRODE = 20

df_train_completo = pd.concat([X_train, y_train], axis=1)
frodi_train = df_train_completo[df_train_completo['isFraud'] == 1]
non_frodi_train = df_train_completo[df_train_completo['isFraud'] == 0]

n_lecite_da_tenere = min(len(non_frodi_train), len(frodi_train) * RATIO_LECITE_PER_FRODE)
non_frodi_campionate = non_frodi_train.sample(n=n_lecite_da_tenere, random_state=42)

df_train_ridotto = pd.concat([frodi_train, non_frodi_campionate]).sample(frac=1, random_state=42)

X_train = df_train_ridotto.drop('isFraud', axis=1)
y_train = df_train_ridotto['isFraud']

print(f"--- CONTEGGIO TRAINING SET RIDOTTO (rapporto 1:{RATIO_LECITE_PER_FRODE}) ---")
print(y_train.value_counts())


# PREPROCESSING
# Standardizzo le colonne numeriche (comprese quelle ingegnerizzate) e codifico quella
# categorica in variabili dummy

categorical_cols = ['type']
numeric_cols = ['amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest',
                 'errorBalanceOrig', 'errorBalanceDest']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(drop='first'), categorical_cols)
    ]
)


# Applico il preprocessor: lo "alleno" sul training set ridotto...
X_train_processed = preprocessor.fit_transform(X_train)

# ...e lo applico a validation e test set senza modificarlo, per simulare dati mai visti
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

--- CONTEGGIO TRAINING SET RIDOTTO (rapporto 1:20) ---
isFraud
0    98560
1     4928
Name: count, dtype: int64


In [23]:
#@title Conversione in tensori PyTorch
import torch
from torch.utils.data import TensorDataset, DataLoader

# CONVERSIONE IN MATRICI DENSE
# L'OneHotEncoder può restituire una matrice sparsa: la convertiamo in densa con .toarray(),
# perché PyTorch lavora con array "normali"
if hasattr(X_train_processed, "toarray"):
    X_train_dense = X_train_processed.toarray()
else:
    X_train_dense = X_train_processed

if hasattr(X_val_processed, "toarray"):
    X_val_dense = X_val_processed.toarray()
else:
    X_val_dense = X_val_processed

if hasattr(X_test_processed, "toarray"):
    X_test_dense = X_test_processed.toarray()
else:
    X_test_dense = X_test_processed


# CREAZIONE DEI TENSORI PYTORCH

# Tensori di input e target per il TRAINING
X_train_tensor = torch.tensor(X_train_dense, dtype=torch.float32)
# .values estrae l'array numpy puro da y_train, necessario per costruire il tensore
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).unsqueeze(1)

# Tensori di input e target per il VALIDATION (usati per l'early stopping e la scelta della soglia)
X_val_tensor = torch.tensor(X_val_dense, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val.values, dtype=torch.float32).unsqueeze(1)

# Tensori di input e target per il TEST
X_test_tensor = torch.tensor(X_test_dense, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).unsqueeze(1)


# CREAZIONE DEL DATALOADER
# Il DataLoader passa i dati al modello a piccoli gruppi (batch) durante l'addestramento.
# Batch più grande di prima (4096 invece di 2048): con un training set molto più piccolo
# (~100k righe invece di 3,8M) riduce il numero di iterazioni per epoca e velocizza il training.
batch_size = 4096
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

print("Tensori e DataLoader creati con successo!")
print(f"Dimensioni X_train_tensor: {X_train_tensor.shape}")
print(f"Dimensioni y_train_tensor: {y_train_tensor.shape}")
print(f"Dimensioni X_val_tensor: {X_val_tensor.shape}")

Tensori e DataLoader creati con successo!
Dimensioni X_train_tensor: torch.Size([103488, 11])
Dimensioni y_train_tensor: torch.Size([103488, 1])
Dimensioni X_val_tensor: torch.Size([1272524, 11])


In [24]:
#@title Definizione della rete neurale
class FraudDetectionNN(nn.Module):
    def __init__(self, input_dim):
        super(FraudDetectionNN, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.2), # Dropout più alto nei primi strati, per limitare l'overfitting
            
            nn.Linear(32, 16),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.Dropout(0.1), # Dropout più leggero verso l'uscita
            
            nn.Linear(16, 1)
        )
        
    def forward(self, x):
        return self.network(x)

input_dim = X_train_tensor.shape[1]
model = FraudDetectionNN(input_dim)
model.to(device)

FraudDetectionNN(
  (network): Sequential(
    (0): Linear(in_features=11, out_features=32, bias=True)
    (1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=32, out_features=16, bias=True)
    (5): BatchNorm1d(16, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.1, inplace=False)
    (8): Linear(in_features=16, out_features=1, bias=True)
  )
)

In [25]:
#@title Configurazione addestramento

# Peso della classe frode nella loss: dato che non facciamo più under-sampling,
# uso pos_weight per dire alla rete che sbagliare una frode (falso negativo) pesa
# quanto (num_lecite / num_frodi) errori sulle transazioni lecite
num_neg = (y_train == 0).sum()
num_pos = (y_train == 1).sum()
pos_weight_value = num_neg / num_pos
pos_weight = torch.tensor([pos_weight_value], dtype=torch.float32).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# Optimizer Adam. Learning rate alzato rispetto a prima (0.0001 era troppo basso:
# dopo 100 epoche la loss non si era ancora stabilizzata)
optimizer = optim.Adam(model.parameters(), lr=0.001)

print(f"pos_weight calcolato per bilanciare la classe frode: {pos_weight_value:.1f}")

pos_weight calcolato per bilanciare la classe frode: 20.0


In [26]:
#@title Training loop
epochs = 100
patience = 10  # numero di epoche senza miglioramenti dopo cui fermarsi

best_val_loss = float('inf')
epochs_no_improve = 0
best_model_state = None

X_val_device = X_val_tensor.to(device)
y_val_device = y_val_tensor.to(device)

print("Inizio Addestramento...")
for epoch in range(epochs):
    model.train()  # Modalità training: attiva Dropout e BatchNorm
    running_loss = 0.0
    
    for inputs, labels in train_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()  # Azzero i gradienti del batch precedente
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()  # Calcolo i gradienti
        optimizer.step()  # Aggiorno i pesi del modello
        
        running_loss += loss.item()
        
    avg_train_loss = running_loss / len(train_loader)

    # Valuto la loss sul validation set, per capire se il modello sta ancora migliorando
    model.eval()
    with torch.no_grad():
        val_outputs = model(X_val_device)
        val_loss = criterion(val_outputs, y_val_device).item()

    print(f"Epoca [{epoch+1}/{epochs}], Loss Training: {avg_train_loss:.4f}, Loss Validation: {val_loss:.4f}")

    # EARLY STOPPING: salvo i pesi solo quando migliora la loss di validation,
    # e mi fermo se non migliora per troppe epoche di fila (evita overfitting)
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_no_improve = 0
        best_model_state = {k: v.clone() for k, v in model.state_dict().items()}
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f"\nEarly stopping all'epoca {epoch+1}: nessun miglioramento su validation da {patience} epoche.")
            break

# Ripristino i pesi del modello all'epoca con la validation loss migliore
if best_model_state is not None:
    model.load_state_dict(best_model_state)
    print(f"Ripristinati i pesi migliori (Loss Validation: {best_val_loss:.4f})")

Inizio Addestramento...
Epoca [1/100], Loss Training: 1.2880, Loss Validation: 0.6783
Epoca [2/100], Loss Training: 1.0510, Loss Validation: 0.6326
Epoca [3/100], Loss Training: 0.8955, Loss Validation: 0.5721
Epoca [4/100], Loss Training: 0.7739, Loss Validation: 0.5054
Epoca [5/100], Loss Training: 0.6806, Loss Validation: 0.4403
Epoca [6/100], Loss Training: 0.6082, Loss Validation: 0.3852
Epoca [7/100], Loss Training: 0.5474, Loss Validation: 0.3425
Epoca [8/100], Loss Training: 0.4982, Loss Validation: 0.3033
Epoca [9/100], Loss Training: 0.4533, Loss Validation: 0.2688
Epoca [10/100], Loss Training: 0.4197, Loss Validation: 0.2425
Epoca [11/100], Loss Training: 0.3894, Loss Validation: 0.2230
Epoca [12/100], Loss Training: 0.3669, Loss Validation: 0.1975
Epoca [13/100], Loss Training: 0.3473, Loss Validation: 0.1780
Epoca [14/100], Loss Training: 0.3306, Loss Validation: 0.1624
Epoca [15/100], Loss Training: 0.3150, Loss Validation: 0.1537
Epoca [16/100], Loss Training: 0.3018, L

In [29]:
#@title Valutazione del modello
model.eval()  # Modalità valutazione: disattiva Dropout e BatchNorm

with torch.no_grad():
    # Probabilità sul validation set, per trovare la soglia ottimale
    val_probs = torch.sigmoid(model(X_val_tensor.to(device))).cpu().numpy().ravel()

# Trovo la soglia che massimizza l'F1 sulla classe frode, usando il validation set
# (prima la soglia era fissata arbitrariamente a 0.8, troppo alta per un modello che
# non era arrivato a convergenza: penalizzava moltissimo il recall)
precisions, recalls, thresholds = precision_recall_curve(y_val, val_probs)
f1_scores = 2 * precisions * recalls / (precisions + recalls + 1e-9)
best_idx = f1_scores[:-1].argmax()
best_threshold = thresholds[best_idx]
print(f"Soglia ottimale trovata su validation (F1 massimo): {best_threshold:.3f}")

X_test_device = X_test_tensor.to(device)
y_test_device = y_test_tensor.to(device)

with torch.no_grad():
    # 1. Calcolo le predizioni sui dati di test
    test_outputs = model(X_test_device)
    
    # 2. Applico la sigmoide per trasformare l'output in una probabilità tra 0 e 1
    probabilities = torch.sigmoid(test_outputs)
    
    # 3. Classifico come frode usando la soglia ottimizzata sul validation set
    y_pred_tensor = (probabilities >= best_threshold).float()
    
    # 4. Calcolo l'accuratezza finale sul test set
    corrette = (y_pred_tensor == y_test_device).sum().item()
    totali = y_test_device.size(0)
    accuratezza_finale = corrette / totali
    print(f"\nACCURATEZZA GLOBALE SUL TEST SET: {accuratezza_finale:.2%}")
    
    # Per usare scikit-learn (matrice di confusione/report) servono array numpy 1D su CPU
    y_pred = y_pred_tensor.cpu().numpy().astype(int).ravel()

print(f"\nAccuratezza Finale sul Test Set: {accuratezza_finale * 100:.2f}%")

# Matrice di confusione
cm = confusion_matrix(y_test, y_pred)

# La trasformo in un DataFrame con etichette leggibili per righe e colonne
cm_con_titoli = pd.DataFrame(cm, 
                             columns=['PREDETTO LECITO', 'PREDETTO FRODE'], 
                             index=['REALE LECITO', 'REALE FRODE'])

# Stampo a schermo la matrice formattata
print("\nMATRICE DI CONFUSIONE:")
print(cm_con_titoli)

print("\nReport di Classificazione:")
print(classification_report(y_test, y_pred))


Soglia ottimale trovata su validation (F1 massimo): 0.919

ACCURATEZZA GLOBALE SUL TEST SET: 99.94%

Accuratezza Finale sul Test Set: 99.94%

MATRICE DI CONFUSIONE:
              PREDETTO LECITO  PREDETTO FRODE
REALE LECITO          1270737             145
REALE FRODE               584            1058

Report di Classificazione:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00   1270882
           1       0.88      0.64      0.74      1642

    accuracy                           1.00   1272524
   macro avg       0.94      0.82      0.87   1272524
weighted avg       1.00      1.00      1.00   1272524



In [28]:
#@title Salvataggio modello e preprocessor
import joblib
import torch

# 1. Salvo il preprocessor (contiene lo StandardScaler e l'OneHotEncoder già addestrati)
joblib.dump(preprocessor, 'preprocessor.pkl')

# 2. Salvo i pesi della rete neurale addestrata
torch.save(model.state_dict(), 'fraud_detection_pytorch.pth')

print("✅ File 'preprocessor.pkl' e 'fraud_detection_pytorch.pth' creati e salvati con successo!")

✅ File 'preprocessor.pkl' e 'fraud_detection_pytorch.pth' creati e salvati con successo!
